Below is a practical Python implementation of a simplified FOC (Field Oriented Control) motor controller using ADRC (Active Disturbance Rejection Control) concepts for current and speed regulation.

This example is suitable for:

PMSM / BLDC motors

Simulation or embedded Linux SBC

Rapid prototyping before porting to C/C++ or MCU firmware

Learning ADRC + FOC structure

It includes:

Clarke transform

Park transform

Inverse Park

Simple SVPWM voltage generation

ADRC current controllers (Id/Iq)

ADRC speed controller

Motor state loop

For real hardware, replace:

simulated motor feedback

PWM output

ADC sampling

encoder interface


with actual drivers.

---

High-Level FOC Structure

Speed Ref
   ↓
Speed ADRC
   ↓ iq_ref
Current ADRC (Id/Iq)
   ↓ vd/vq
Inverse Park
   ↓
SVPWM
   ↓
3-Phase Inverter
   ↓
Motor


---

#Python FOC + ADRC Controller

In [1]:
pip install numpy


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import math
import time

# ============================================================
# Utility Functions
# ============================================================

In [3]:
# ============================================================
# Utility Functions
# ============================================================

SQRT3 = np.sqrt(3.0)


def clamp(x, xmin, xmax):
    return max(xmin, min(x, xmax))


# ============================================================
# Clarke Transform
# ============================================================

def clarke(ia, ib, ic):
    alpha = ia
    beta = (ia + 2 * ib) / SQRT3
    return alpha, beta


# ============================================================
# Park Transform
# ============================================================

def park(alpha, beta, theta):
    c = math.cos(theta)
    s = math.sin(theta)

    d = alpha * c + beta * s
    q = -alpha * s + beta * c

    return d, q


# ============================================================
# Inverse Park
# ============================================================

def inverse_park(vd, vq, theta):
    c = math.cos(theta)
    s = math.sin(theta)

    alpha = vd * c - vq * s
    beta = vd * s + vq * c

    return alpha, beta


# ============================================================
# SVPWM
# ============================================================

def svpwm(alpha, beta, vdc):

    va = alpha
    vb = -0.5 * alpha + (SQRT3 / 2.0) * beta
    vc = -0.5 * alpha - (SQRT3 / 2.0) * beta

    duty_a = 0.5 + va / vdc
    duty_b = 0.5 + vb / vdc
    duty_c = 0.5 + vc / vdc

    duty_a = clamp(duty_a, 0.0, 1.0)
    duty_b = clamp(duty_b, 0.0, 1.0)
    duty_c = clamp(duty_c, 0.0, 1.0)

    return duty_a, duty_b, duty_c

# ============================================================
# ADRC Controller
# ============================================================

In [4]:
# ============================================================
# ADRC Controller
# ============================================================

class ADRC:

    def __init__(self, b0, kp, kd, dt):

        self.b0 = b0
        self.kp = kp
        self.kd = kd
        self.dt = dt

        # ESO states
        self.z1 = 0.0
        self.z2 = 0.0
        self.z3 = 0.0

        # ESO gains
        self.beta1 = 100.0
        self.beta2 = 3000.0
        self.beta3 = 20000.0

    def update(self, ref, y):

        e = self.z1 - y

        # Extended State Observer
        self.z1 += self.dt * (self.z2 - self.beta1 * e)
        self.z2 += self.dt * (self.z3 - self.beta2 * e)
        self.z3 += self.dt * (-self.beta3 * e)

        # Tracking error
        err = ref - self.z1

        # PD control
        u0 = self.kp * err - self.kd * self.z2

        # Disturbance compensation
        u = (u0 - self.z3) / self.b0

        return u

# ============================================================
# Motor Controller
# ============================================================

In [5]:
# ============================================================
# Motor Controller
# ============================================================

class FOCController:

    def __init__(self, dt):

        self.dt = dt

        # Current loop ADRC
        self.id_ctrl = ADRC(
            b0=20.0,
            kp=5.0,
            kd=0.02,
            dt=dt
        )

        self.iq_ctrl = ADRC(
            b0=20.0,
            kp=5.0,
            kd=0.02,
            dt=dt
        )

        # Speed loop ADRC
        self.speed_ctrl = ADRC(
            b0=5.0,
            kp=0.5,
            kd=0.01,
            dt=dt
        )

        self.id_ref = 0.0

    def update(
        self,
        speed_ref,
        speed_meas,
        ia,
        ib,
        ic,
        theta,
        vdc
    ):

        # ====================================================
        # Clarke Transform
        # ====================================================

        alpha, beta = clarke(ia, ib, ic)

        # ====================================================
        # Park Transform
        # ====================================================

        id_meas, iq_meas = park(alpha, beta, theta)

        # ====================================================
        # Speed ADRC
        # ====================================================

        iq_ref = self.speed_ctrl.update(
            speed_ref,
            speed_meas
        )

        # ====================================================
        # Current ADRC
        # ====================================================

        vd = self.id_ctrl.update(
            self.id_ref,
            id_meas
        )

        vq = self.iq_ctrl.update(
            iq_ref,
            iq_meas
        )

        # Voltage limiting
        vmax = vdc * 0.577

        vd = clamp(vd, -vmax, vmax)
        vq = clamp(vq, -vmax, vmax)

        # ====================================================
        # Inverse Park
        # ====================================================

        alpha_v, beta_v = inverse_park(
            vd,
            vq,
            theta
        )

        # ====================================================
        # SVPWM
        # ====================================================

        duty_a, duty_b, duty_c = svpwm(
            alpha_v,
            beta_v,
            vdc
        )

        return {
            "duty_a": duty_a,
            "duty_b": duty_b,
            "duty_c": duty_c,
            "id": id_meas,
            "iq": iq_meas,
            "vd": vd,
            "vq": vq,
            "iq_ref": iq_ref
        }


# ============================================================
# Simulated Motor
# ============================================================

In [6]:
# ============================================================
# Simulated Motor
# ============================================================

class SimpleMotor:

    def __init__(self):

        self.theta = 0.0
        self.speed = 0.0

    def update(self, torque_cmd, dt):

        # Simple motor dynamics
        J = 0.01
        B = 0.001

        accel = (torque_cmd - B * self.speed) / J

        self.speed += accel * dt
        self.theta += self.speed * dt

        ia = math.sin(self.theta)
        ib = math.sin(self.theta - 2.094)
        ic = math.sin(self.theta + 2.094)

        return ia, ib, ic



# ============================================================
# Main Loop
# ============================================================

In [7]:
# ============================================================
# Main Loop
# ============================================================

if __name__ == "__main__":

    dt = 0.001

    foc = FOCController(dt)
    motor = SimpleMotor()

    speed_ref = 100.0
    vdc = 24.0

    for i in range(5000):

        ia, ib, ic = motor.update(
            torque_cmd=1.0,
            dt=dt
        )

        result = foc.update(
            speed_ref=speed_ref,
            speed_meas=motor.speed,
            ia=ia,
            ib=ib,
            ic=ic,
            theta=motor.theta,
            vdc=vdc
        )

        if i % 100 == 0:
            print(
                f"Speed: {motor.speed:.2f}, "
                f"Iq_ref: {result['iq_ref']:.2f}, "
                f"DutyA: {result['duty_a']:.2f}"
            )

        time.sleep(dt)

Speed: 0.10, Iq_ref: 9.60, DutyA: 0.50
Speed: 10.05, Iq_ref: -91.52, DutyA: 0.78
Speed: 19.90, Iq_ref: -34.11, DutyA: 0.85
Speed: 29.65, Iq_ref: -8.54, DutyA: 0.41
Speed: 39.31, Iq_ref: 0.98, DutyA: 0.49
Speed: 48.87, Iq_ref: 4.15, DutyA: 0.51
Speed: 58.33, Iq_ref: 4.82, DutyA: 0.55
Speed: 67.70, Iq_ref: 4.51, DutyA: 0.55
Speed: 76.98, Iq_ref: 3.82, DutyA: 0.51
Speed: 86.16, Iq_ref: 2.98, DutyA: 0.46
Speed: 95.26, Iq_ref: 2.10, DutyA: 0.53
Speed: 104.26, Iq_ref: 1.19, DutyA: 0.48
Speed: 113.17, Iq_ref: 0.29, DutyA: 0.50
Speed: 122.00, Iq_ref: -0.60, DutyA: 0.50
Speed: 130.73, Iq_ref: -1.49, DutyA: 0.50
Speed: 139.38, Iq_ref: -2.37, DutyA: 0.51
Speed: 147.95, Iq_ref: -3.24, DutyA: 0.52
Speed: 156.43, Iq_ref: -4.11, DutyA: 0.47
Speed: 164.82, Iq_ref: -4.96, DutyA: 0.53
Speed: 173.13, Iq_ref: -5.81, DutyA: 0.51
Speed: 181.36, Iq_ref: -6.65, DutyA: 0.45
Speed: 189.51, Iq_ref: -7.48, DutyA: 0.44
Speed: 197.57, Iq_ref: -8.30, DutyA: 0.45
Speed: 205.55, Iq_ref: -9.11, DutyA: 0.55
Speed: 213.4

---

What This Implements

FOC Core

Clarke Transform

Park Transform

dq current control

Inverse Park

SVPWM duty generation



---

ADRC Features

The ADRC observer estimates:

load disturbance

back EMF effects

parameter mismatch

friction

external torque disturbances


ESO structure:

z1 = estimated state
z2 = estimated derivative
z3 = estimated disturbance

Control law:

u = (u0 - z3)/b0

This gives much better robustness than PID.


---

Real Hardware Integration

Replace these sections:

ADC Current Read

ia, ib, ic

with:

shunt current ADC

inline current sensors

Hall current sensors



---

Rotor Position

theta

from:

encoder

resolver

Hall sensors

sensorless estimator



---

PWM Output

Replace:

return duty_a, duty_b, duty_c

with:

STM32 TIM1/TIM8

ESP32 MCPWM

Linux PWM

FPGA PWM core



---

Recommended Real-Time Architecture

For real hardware:

Loop	Frequency

Current Loop	10–40 kHz
Speed Loop	1–5 kHz
Position Loop	100–1000 Hz


Python alone is usually too slow for hard real-time FOC on Linux without:

PREEMPT_RT

Xenomai

FPGA offload

MCU coprocessor



---

Better Production Architecture

Recommended stack:

Layer	Technology

Fast current loop	STM32 / FPGA / RP2350
High-level ADRC tuning	Python
Visualization	PyQt / Web UI
ML tuning	PyTorch
EtherCAT/CAN	SOEM / SocketCAN



---

Advanced Improvements

You can extend this with:

Sensorless observer

Sliding mode observer

MTPA

Field weakening

Space vector overmodulation

Flux observer

Dead-time compensation

Anti-cogging

Cascaded ADRC

Nonlinear ADRC (NLSEF)

EKF rotor estimator

FPGA SVPWM accelerator



---

Recommended Hardware

For serious implementation:

MCU

STMicroelectronics STM32G4 / H7

Texas Instruments C2000

Espressif Systems ESP32-S3


FPGA

AMD Zynq RFSoC

Intel Cyclone V


Motor Driver IC

Texas Instruments DRV8302

Infineon Technologies 6EDL7141

STMicroelectronics STSPIN32